# Measuring Plant Disease Severity with YOLO Instance Segmentation

## Introduction: the problem

Plant pathologists and breeders need to know not only **whether** a plant is diseased, but
**how badly**. **Disease severity** is usually expressed as the *percentage of the leaf area that
is diseased*. It drives decisions such as when to spray, how to rank resistant varieties and how
to estimate yield loss. Traditionally severity is estimated **by eye**, sometimes with the help
of standard area diagrams. This is slow, and different raters give different scores for the same
leaf.

Image analysis can make severity measurement **objective and fast**. To measure an *area*,
however, a model must know exactly **which pixels** belong to the leaf and which belong to the
lesion. A bounding box is not enough. That is what **instance segmentation** provides: for every
object it outputs a class, a box, and a pixel-precise **mask**.

### The crops and problems in this dataset

| Crop | Class | What it is |
|---|---|---|
| Maize | `maize leaf` | Leaf area (healthy tissue + lesions) |
| Maize | `maize leaf blight` | Fungal leaf blight lesions (e.g. northern corn leaf blight, *Exserohilum turcicum*): long, gray-brown, cigar-shaped lesions |
| Maize | `maize fall armyworm` | Feeding damage by the insect pest *Spodoptera frugiperda*: ragged holes and "windows" in the leaves |
| Potato | `potato leaf` | Leaf area |
| Potato | `potato late blight` | Lesions of *Phytophthora infestans* (an oomycete; the cause of the Irish potato famine): dark, water-soaked, fast-spreading lesions |
| Potato | `potato pest damage` | Insect feeding damage |
| Tomato | `tomato leaf` | Leaf area |
| Tomato | `tomato early blight` | Fungal lesions (*Alternaria* spp.): brown spots with concentric rings ("target spots") |
| Tomato | `tomato leaf miner` | Mines (tunnels) made by larvae of the moth *Tuta absoluta* |

Each crop has one **leaf** class (the plant surface) and two **damage** classes (disease or pest).
This lets us compute severity as *damaged area ÷ leaf area*.

### What will the model do?
For every photo the model will output a **mask** for every leaf and every lesion or damaged
region, each with a class name and a confidence. From those masks we compute the
**percentage of damaged leaf area** and compare it with the value computed from the human
annotations.

### The data
**AgriVision / plantdisease** (Kaggle: https://www.kaggle.com/datasets/social888/plantdisease,
about 2,000 images, about 2.1 GB). The dataset is a **Roboflow YOLOv8 segmentation export**: the
labels are already polygons in YOLO format, so no format conversion is needed.

Things to keep in mind:
- Roboflow exports often contain **augmented copies** of the same photo (file names like
  `photo_jpg.rf.<code>.jpg`). We keep all copies of a photo in the **same** split, otherwise the
  model would be tested on near-copies of its training images.
- The Kaggle version may ship only a `train/` split. Missing validation and test splits are
  created in Step 5.
- Leaves are large and lesions are small, so expect much better scores for leaf classes than for
  damage classes.

### Learning objectives
By the end of this notebook you should be able to:
1. Explain the difference between classification, object detection and instance segmentation.
2. Prepare a segmentation dataset (polygon labels, leakage-free train/val/test split).
3. Train a YOLO segmentation model using **transfer learning**.
4. Evaluate it with **box and mask mAP**, overall and per class.
5. Use the predicted masks to **estimate disease severity** and check it against the annotations.

### Roadmap
| Step | What happens |
|---|---|
| 1 | Set up Colab (GPU, libraries) |
| 2 | Download the dataset from Google Drive and unzip it |
| 3 | Inspect classes, splits and label format |
| 4 | (Optional) Keep only selected crops |
| 5 | Build the train / validation / test dataset |
| 6 | Look at the ground-truth masks |
| 7 | Train the model (**settings you can change are here**) |
| 8 | Inspect the training curves |
| 9 | Evaluate on the validation and **test** sets |
| 10 | Predict masks on test images |
| 11 | Post-analysis: disease severity (% damaged leaf area) |
| 12 | Log the experiment |
| Final | Instructions for your report |

> Throughout the notebook, notes marked **For your report** tell you what to write down.

## Key concepts

| Term | Meaning |
|---|---|
| **Classification** | One label for the whole image ("this is late blight") |
| **Object detection** | A box + class for every object ("a lesion is *here*") |
| **Instance segmentation** | A box + class + pixel **mask** for every object ("these exact pixels are lesion") |
| **Polygon label** | How masks are stored in YOLO format: `class x1 y1 x2 y2 x3 y3 ...`, the outline of the object as normalised (0-1) points |
| **IoU** (Intersection over Union) | Overlap between prediction and truth (0-1). For masks it is computed on **pixels**, for boxes on rectangles |
| **Precision / Recall** | Precision: fraction of predictions that are correct (*false alarms?*). Recall: fraction of true objects that were found (*misses?*) |
| **mAP50 / mAP50-95** | Average precision over all classes at IoU ≥ 0.5 / averaged over IoU 0.5…0.95 (stricter) |
| **(B) vs (M)** | Ultralytics reports every metric twice: **B**ox (are the objects found and boxed?) and **M**ask (are the pixels right?) |
| **Severity (%)** | Damaged area ÷ leaf area × 100, the standard way to quantify disease in plant pathology |
| **Transfer learning** | Start from a model pretrained on the COCO dataset and fine-tune it on our images |
| **Train / validation / test** | *Train*: the model learns from these. *Validation*: used during training to pick the best epoch. *Test*: never seen during training, so it gives an unbiased final score |

# 1. Setup

### 1.1 Turn on a GPU
In Colab, open `Runtime > Change runtime type`, choose **T4 GPU** (or any GPU) and click `Save`.
Segmentation training on a CPU is far too slow for this dataset.

### 1.2 Install and import
The next cells check the GPU, install the **Ultralytics** library (which provides YOLO) and
import the Python packages used in the notebook. If `nvidia-smi` prints an error, you are not on
a GPU runtime.

In [ ]:
!nvidia-smi

**Install Ultralytics**, the library that provides YOLO. `-U` upgrades it to the latest version,
so the newest YOLO models (YOLO26) are available.

In [ ]:
!pip install -q -U ultralytics

**Import the libraries** used in the notebook and choose the device: GPU 0 if one is available,
otherwise the CPU. The cell prints the device that will be used for training.

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display
import os, glob, random, shutil, re, yaml, zipfile
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import torch

DEVICE = 0 if torch.cuda.is_available() else "cpu"   # GPU 0 if available, else CPU
print("Ultralytics YOLO loaded. Training device:", DEVICE)

# 2. Get the data

### 2.1 Download the dataset from Google Drive

The dataset is one zip file of about 2.1 GB (`archive.zip`, the Roboflow YOLOv8 segmentation
export of the Kaggle dataset described in the introduction). It has been shared from Google Drive
with a link, so you do not need to download it to your computer or sign in anywhere.

The cell below:
- keeps the share link in `DATA_URL`;
- downloads the file straight into Colab, to `/content/dataset.zip`, with **gdown** (a small tool
  that downloads files from Google Drive share links);
- skips the download if the file is already there (for example if you run the cell twice);
- checks that the downloaded file is a complete zip archive and prints its size.

The download usually takes 1 to 3 minutes.

> **If the download fails** (for example a message that the file has been downloaded too many
> times), wait a while and run the cell again, or ask your instructor. As a last resort, download
> the zip in your browser from the link in `DATA_URL`, drag it into `/content` with the **Files**
> panel (left sidebar), and continue with Step 2.2.

In [ ]:
DATA_URL = "https://drive.google.com/file/d/1oxt6plsaVXfJUAsXc8UgoPqo4Dz5Fo4-/view?usp=sharing"
ARCHIVE  = "/content/dataset.zip"

if not os.path.exists(ARCHIVE):
    !pip install -q gdown
    !gdown --fuzzy "{DATA_URL}" -O {ARCHIVE}

if not (os.path.exists(ARCHIVE) and zipfile.is_zipfile(ARCHIVE)):   # fallback: a zip uploaded by hand
    uploaded = [z for z in sorted(glob.glob("/content/*.zip")) if zipfile.is_zipfile(z)]
    assert uploaded, "Download failed and no zip found in /content - see the note above."
    ARCHIVE = uploaded[0]
print(f"{ARCHIVE}: {os.path.getsize(ARCHIVE) / 1e9:.2f} GB")

### 2.2 Unzip the dataset

The cell extracts the zip into `/content/data` (this takes a minute or two) and then finds the
**dataset root**: the folder that contains `data.yaml`, the file that lists the classes and the
image folders. It prints the location of the dataset root.

In [ ]:
print("Extracting", ARCHIVE, "...")
with zipfile.ZipFile(ARCHIVE) as z:
    z.extractall("/content/data")
print("Done.")

# dataset root = the (shallowest) folder that contains data.yaml; skip __MACOSX copies
yamls = [p for p in glob.glob("/content/data/**/data.yaml", recursive=True) if "__MACOSX" not in p]
assert yamls, "No data.yaml found after unzip - is this the YOLOv8 export?"
DATA_ROOT = os.path.dirname(sorted(yamls, key=len)[0])
print("DATA_ROOT =", DATA_ROOT)

# 3. Inspect the dataset

Before training, we check three things:
1. **The classes** listed in `data.yaml`.
2. **Which splits exist** (`train`, `valid`, `test`) and how many images each has.
3. **The label format.** A segmentation label line is `class x1 y1 x2 y2 …` (many numbers: the
   polygon outline). A detection label line is just `class xc yc w h` (4 numbers).

We also count the **instances (objects) per class**, which you need for the dataset description.

> **For your report (Methodology):** record the number of images, the classes and the number
> of instances per class. Which classes are rare?

In [ ]:
with open(os.path.join(DATA_ROOT, "data.yaml")) as f:
    ycfg = yaml.safe_load(f)
names = ycfg.get("names")
if isinstance(names, dict):
    names = [names[k] for k in sorted(names)]
print(f"{len(names)} classes:", names)

# which splits physically exist (a folder with an images/ subdir)?
SRC_SPLITS = [s for s in ("train", "valid", "val", "test")
              if os.path.isdir(os.path.join(DATA_ROOT, s, "images"))]
IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

def label_of(img_path):
    """YOLO label file belonging to an image (…/images/x.jpg -> …/labels/x.txt)."""
    p = Path(img_path)
    return str(p.parent.parent / "labels" / (p.stem + ".txt"))

def read_polys(label_path):
    """List of (class_id, Nx2 array of normalised points); skips non-polygon lines."""
    out = []
    if os.path.exists(label_path):
        for line in open(label_path):
            t = line.split()
            if len(t) >= 7 and len(t) % 2 == 1:          # class + >=3 (x, y) points
                out.append((int(float(t[0])), np.array(t[1:], float).reshape(-1, 2)))
    return out

src_images = {s: sorted(p for p in glob.glob(os.path.join(DATA_ROOT, s, "images", "*"))
                        if p.lower().endswith(IMG_EXTS)) for s in SRC_SPLITS}
print("splits found (images):", {s: len(v) for s, v in src_images.items()})

# label format check + instance counts
n_poly = n_box = 0
inst = Counter()
for s in SRC_SPLITS:
    for img in src_images[s]:
        lp = label_of(img)
        if not os.path.exists(lp):
            continue
        for line in open(lp):
            t = line.split()
            if len(t) == 5: n_box += 1
            elif len(t) > 5: n_poly += 1; inst[names[int(float(t[0]))]] += 1
print(f"label lines: {n_poly} polygons, {n_box} plain boxes "
      f"-> {'segmentation format' if n_poly > n_box else 'WARNING: looks like detection labels'}")
pd.DataFrame({"instances": pd.Series(inst)}).reindex(names).fillna(0).astype(int)

# 4. (Optional) Keep only selected crops

Training on all three crops gives the most data. You may, however, want to study **one crop**
(for example only tomato) or compare crops in separate experiments. The dataset ships a
`metadata.csv` describing each image. The first cell reads it, detects the relevant columns and
shows how many **healthy** and **sick** images each crop has.

Then set `SELECT_CROPS` in the second cell. Leave it as `None` to use the **full dataset**, in
which case this step does nothing.

In [ ]:
def base_of(stem):
    """Original photo name: strips Roboflow's '_jpg.rf.<hash>' suffix from augmented copies."""
    return re.sub(r"_(jpe?g|png|bmp|webp)\.rf\.[0-9a-zA-Z]+$", "", stem, flags=re.I)

mp = [p for p in glob.glob(os.path.join(DATA_ROOT, "**", "metadata.csv"), recursive=True)
      if "__MACOSX" not in p]
meta = None
if not mp:
    print("No metadata.csv found - crop filter will be skipped (full dataset).")
else:
    meta = pd.read_csv(mp[0])
    print("metadata.csv columns:", list(meta.columns))
    print(meta.head(3).to_string(), "\n")

    cl = {c.lower(): c for c in meta.columns}
    def pick(*keys):
        for k in keys:
            for lc, o in cl.items():
                if k in lc:
                    return o
        return None
    FILE_COL   = pick("file", "image", "name", "path")
    CROP_COL   = pick("crop", "species", "plant")
    LABEL_COL  = pick("label", "class", "disease", "category")
    HEALTH_COL = pick("health", "status", "condition")
    print("detected -> file:", FILE_COL, "| crop:", CROP_COL,
          "| label:", LABEL_COL, "| health:", HEALTH_COL)

    def _crop(r):
        if CROP_COL:  return str(r[CROP_COL]).strip().lower()
        if LABEL_COL: return str(r[LABEL_COL]).replace("_", " ").split()[0].lower()
        return "unknown"
    def _health(r):
        if HEALTH_COL is not None:                      # e.g. is_healthy = 1/0
            v = str(r[HEALTH_COL]).strip().lower()
            if v in ("1", "true", "yes", "healthy"): return "healthy"
            if v in ("0", "false", "no", "sick"):    return "sick"
        txt = str(r[LABEL_COL]).lower() if LABEL_COL else ""
        return "healthy" if "healthy" in txt else "sick"
    meta["_crop"]   = meta.apply(_crop, axis=1)
    meta["_health"] = meta.apply(_health, axis=1)
    print("\ncrop x health image counts:")
    print(pd.crosstab(meta["_crop"], meta["_health"], margins=True))

**Choose the crops (edit this cell).** Set `SELECT_CROPS` to a list of crops, for example
`["tomato"]`, to keep only their images, and `INCLUDE_HEALTH` to keep healthy and/or sick images.
With `SELECT_CROPS = None` the full dataset is used. The cell prints how many images are kept.

In [ ]:
# ===== edit these =====
SELECT_CROPS   = None                  # e.g. ["tomato"] or ["tomato", "potato"]; None = keep all
INCLUDE_HEALTH = ["healthy", "sick"]   # keep both by default
# ======================

KEEP = None                            # None = keep every image
if SELECT_CROPS and meta is not None and FILE_COL:
    want = [c.lower() for c in SELECT_CROPS]
    sel  = meta[meta["_crop"].isin(want) & meta["_health"].isin(INCLUDE_HEALTH)]
    keep_bases = {base_of(Path(str(f)).stem) for f in sel[FILE_COL]}
    all_imgs = [p for s in SRC_SPLITS for p in src_images[s]]
    KEEP = {p for p in all_imgs if base_of(Path(p).stem) in keep_bases}       # exact name match
    if not KEEP:   # some exports rename files; fall back to prefix matching
        print("No exact file-name matches - falling back to prefix matching; check the result!")
        KEEP = {p for p in all_imgs
                if any(base_of(Path(p).stem).startswith(b) for b in keep_bases if len(b) >= 4)}
    print(f"selected {len(sel)} metadata rows -> {len(KEEP)} of {len(all_imgs)} images kept")
    assert KEEP, "No images matched - check FILE_COL / the file-name format printed above."
elif SELECT_CROPS:
    print("SELECT_CROPS is set but there is no usable metadata.csv - using the full dataset.")
else:
    print("No filtering (SELECT_CROPS = None) - using the full dataset.")

# 5. Build the train / validation / test dataset

We copy the images and labels into a clean working folder, `/content/seg_data`, with three splits:

- Splits that **already exist** in the download are kept as they are.
- If **validation** and/or **test** is missing, it is carved out of `train`, aiming at roughly
  **70 / 20 / 10** of all images overall.
- The carving is done **per original photo** (`base_of`): all augmented copies of one photo go to
  the same split, so the model is never tested on a near-copy of a training image.
- A fixed random seed (42) and sorted file lists make the split identical for everyone.

The original extracted data is never modified, so you can safely re-run this cell (for example
after changing `SELECT_CROPS`).

> **For your report (Methodology):** the table printed below (images and instances per class
> per split) is your dataset table.

In [ ]:
WORK = "/content/seg_data"
FRAC = {"valid": 0.20, "test": 0.10}           # target fractions of ALL images
random.seed(42)

keep = lambda p: KEEP is None or p in KEEP
pool = {("valid" if s == "val" else s): [p for p in src_images[s] if keep(p)] for s in SRC_SPLITS}
total = sum(len(v) for v in pool.values())

# carve missing splits out of train, grouping augmented copies of the same photo
groups = defaultdict(list)
for p in pool.get("train", []):
    groups[base_of(Path(p).stem)].append(p)
order = sorted(groups); random.shuffle(order)
for s in ("valid", "test"):
    if pool.get(s):
        continue
    target, chosen = int(FRAC[s] * total), []
    while order and len(chosen) < target:
        chosen += groups[order.pop()]
    pool[s] = chosen
    print(f"No '{s}' split in the download -> carved {len(chosen)} images out of train")
pool["train"] = [p for b in order for p in groups[b]]

shutil.rmtree(WORK, ignore_errors=True)
for s, imgs in pool.items():
    for sub in ("images", "labels"):
        os.makedirs(os.path.join(WORK, s, sub), exist_ok=True)
    for p in imgs:
        shutil.copy2(p, os.path.join(WORK, s, "images", Path(p).name))
        if os.path.exists(label_of(p)):
            shutil.copy2(label_of(p), os.path.join(WORK, s, "labels", Path(p).stem + ".txt"))

DATA_YAML = os.path.join(WORK, "data.yaml")
with open(DATA_YAML, "w") as f:
    yaml.safe_dump({"path": WORK, "train": "train/images", "val": "valid/images",
                    "test": "test/images", "names": {i: n for i, n in enumerate(names)}},
                   f, sort_keys=False)

SPLIT_NAMES = ["train", "valid", "test"]
split_imgs = {s: sorted(glob.glob(os.path.join(WORK, s, "images", "*"))) for s in SPLIT_NAMES}
inst_split = {s: Counter(names[c] for p in split_imgs[s] for c, _ in read_polys(label_of(p)))
              for s in SPLIT_NAMES}
split_table = pd.DataFrame(inst_split).reindex(names).fillna(0).astype(int)
split_table.loc["images"] = [len(split_imgs[s]) for s in SPLIT_NAMES]
leak = ({base_of(Path(p).stem) for p in split_imgs["train"]} &
        {base_of(Path(p).stem) for s in ("valid", "test") for p in split_imgs[s]})
print(open(DATA_YAML).read())
print("photos shared between train and valid/test:", len(leak),
      "(0 = no leakage)" if not leak else "(from the original download's own split)")
split_table

# 6. Look at the ground-truth masks

**Always look at your data.** The cell draws the annotated polygons on a few random training
images: filled colour = mask, text = class. Check that the outlines follow the leaves and the
lesions. Notice how many lesions there are and how small they are compared with the leaves.
Re-run the cell to see other images.

In [ ]:
import cv2
import matplotlib.pyplot as plt
%matplotlib inline

PALETTE = [tuple(int(v) for v in c) for c in np.random.default_rng(0).integers(40, 255, (len(names), 3))]

def draw_polys(img_path):
    im = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    H, W = im.shape[:2]
    overlay = im.copy()
    for cls, pts in read_polys(label_of(img_path)):
        pts = (pts * [W, H]).astype(np.int32)
        cv2.fillPoly(overlay, [pts], PALETTE[cls])
        cv2.polylines(im, [pts], True, PALETTE[cls], max(2, W // 400))
        cv2.putText(im, names[cls], tuple(int(v) for v in pts[0]), cv2.FONT_HERSHEY_SIMPLEX,
                    max(0.5, W / 1200), PALETTE[cls], 2)
    return cv2.addWeighted(overlay, 0.4, im, 0.6, 0)

samples = random.sample(split_imgs["train"], k=min(3, len(split_imgs["train"])))
fig, axes = plt.subplots(1, len(samples), figsize=(6 * len(samples), 6), squeeze=False)
for ax, p in zip(axes[0], samples):
    ax.imshow(draw_polys(p)); ax.set_title(Path(p).name[:30], fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

# 7. Train the segmentation model

### 7.1 Experiment settings: **this is the cell you change**

Training starts from YOLO segmentation weights **pretrained on the COCO dataset** (transfer
learning). The network already knows general shapes and textures and only has to learn what
leaves and lesions look like.

| Setting | What it does | Suggested values to try |
|---|---|---|
| `MODEL_NAME` | Model size: **n**ano < **s**mall < **m**edium. Bigger = more capacity, slower, more GPU memory | `yolo26n-seg.pt` (default), `yolo26s-seg.pt` |
| `EPOCHS` | Maximum number of passes over the training data | 30, 50 (default), 100 |
| `PATIENCE` | Early stopping: stop if validation mAP has not improved for this many epochs | 15 (default), 30 |
| `IMGSZ` | Images are resized to this many pixels. Larger helps small lesions but is slower and uses more memory | 640 (default), 768, 1024 |
| `BATCH` | Images per training step. Lower it if you get a *CUDA out of memory* error | 16 (default), 8, 4 |

> **Do a proper experiment:** change **one** setting at a time and compare against your
> baseline. Each run gets its own name (`RUN_NAME`) and folder, so results never overwrite each
> other.
>
> On a Colab T4 GPU the default run on the full dataset takes about 30-60 minutes. Larger
> models or image sizes take longer, and Colab may disconnect very long runs.

In [ ]:
# ======================= EXPERIMENT SETTINGS (edit me) =======================
MODEL_NAME = "yolo26n-seg.pt"   # yolo26n-seg.pt / yolo26s-seg.pt / yolo26m-seg.pt
EPOCHS     = 50
PATIENCE   = 15
IMGSZ      = 640
BATCH      = 16
# ==============================================================================
crops_tag  = "all" if not SELECT_CROPS else "-".join(c.lower() for c in SELECT_CROPS)
RUN_NAME   = f"seg_{crops_tag}_{Path(MODEL_NAME).stem}_e{EPOCHS}_img{IMGSZ}_b{BATCH}"
PROJECT    = "/content/runs/plant_seg"
print("Run name:", RUN_NAME)

### 7.2 Train
**Check the installation.** `yolo checks` prints the software and hardware versions (useful for the *Methodology*).

In [ ]:
!yolo checks

**Train the model.** The cell loads the pretrained weights given in `MODEL_NAME`. If that model is not available in the installed
Ultralytics version, it falls back to the equivalent YOLO11 model and says so. It then trains with the
settings from the experiment-settings cell. During
training, each epoch prints the losses and the validation metrics for boxes (`Box P R mAP50
mAP50-95`) and masks (`Mask P R mAP50 mAP50-95`).

The best epoch, judged on the validation set, is saved as **`best.pt`**, which we use from here
on.

In [ ]:
try:
    model = YOLO(MODEL_NAME)
except Exception as e:                       # YOLO26 needs a recent ultralytics version
    fallback = MODEL_NAME.replace("yolo26", "yolo11")
    print(f"'{MODEL_NAME}' unavailable ({e}); falling back to {fallback}")
    MODEL_NAME = fallback
    model = YOLO(MODEL_NAME)

train_results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,   # early-stop if no improvement for PATIENCE epochs
    device=DEVICE,
    seed=42,
    project=PROJECT,
    name=RUN_NAME,       # each experiment gets its own folder
)

RUN_DIR = Path(model.trainer.save_dir)        # actual folder used (e.g. ..._2 if the name existed)
BEST_PT = RUN_DIR / "weights" / "best.pt"
print("Run folder:", RUN_DIR)

# 8. Training curves

`results.png` plots the training history, one curve per epoch:
- **Losses** (`box_loss`, `seg_loss`, `cls_loss`, `dfl_loss`) for training and validation images.
  They should go down. `seg_loss` measures how wrong the masks are. If the training losses keep
  falling while the validation losses rise, the model is **over-fitting**.
- **Metrics** for boxes `(B)` and masks `(M)`: precision, recall, mAP50, mAP50-95 on the
  validation set. These should rise and level off.

> **For your report (Results):** include this figure for your best run. Did the curves level
> off, or would more epochs have helped? At which epoch did training stop?

In [ ]:
results_img = RUN_DIR / "results.png"
if results_img.exists():
    display(Image(filename=str(results_img), width=1000))
else:
    print("results.png not found in", RUN_DIR)

hist = pd.read_csv(RUN_DIR / "results.csv")
hist.columns = hist.columns.str.strip()
best_row = hist.loc[hist["metrics/mAP50(M)"].idxmax()]
print(f"Epochs run: {len(hist)} | best val mask mAP50 = {best_row['metrics/mAP50(M)']:.3f} "
      f"at epoch {int(best_row['epoch'])}")

# 9. Evaluate on the validation and **test** sets

We reload `best.pt` and score it on the **validation** split (used to pick the best epoch, so
slightly optimistic) and the **test** split (never seen during training, so these are the numbers
to report).

The per-class table gives, for every class, the **box** metrics (is the object found?) and the
**mask** metrics (are its pixels right?). Large objects such as leaves usually score much higher
than small lesions, and mask mAP50-95 is usually the lowest number because it demands precise
outlines.

> **For your report (Results):** the per-class **test** table is your main results table. Which
> classes are segmented well and which poorly? How does that relate to their number of training
> instances (Step 5) and their size?

In [ ]:
model = YOLO(str(BEST_PT))
val_metrics = model.val(data=DATA_YAML, split="val", imgsz=IMGSZ, device=DEVICE,
                        project=PROJECT, name=f"{RUN_NAME}_val")
test_metrics = model.val(data=DATA_YAML, split="test", imgsz=IMGSZ, device=DEVICE,
                         project=PROJECT, name=f"{RUN_NAME}_test")
for tag, m in (("VALIDATION", val_metrics), ("TEST", test_metrics)):
    print(f"{tag:10s} box mAP50 = {m.box.map50:.3f}  mAP50-95 = {m.box.map:.3f} | "
          f"mask mAP50 = {m.seg.map50:.3f}  mAP50-95 = {m.seg.map:.3f}")

**Per-class test table.** For every class: the number of training and test instances, the box
precision, recall and mAP, and the mask precision, recall and mAP. The `ALL` row is the average
over classes. The table is also saved as `per_class_test_metrics.csv` in the run folder.

In [ ]:
rows = []
for i, cid in enumerate(test_metrics.box.ap_class_index):
    n = test_metrics.names[int(cid)]
    bp, br, b50, b5095 = test_metrics.box.class_result(i)
    mp_, mr, m50, m5095 = test_metrics.seg.class_result(i)
    rows.append({"class": n, "train_inst": split_table.loc[n, "train"],
                 "test_inst": split_table.loc[n, "test"],
                 "box_P": bp, "box_R": br, "box_mAP50": b50, "box_mAP50-95": b5095,
                 "mask_P": mp_, "mask_R": mr, "mask_mAP50": m50, "mask_mAP50-95": m5095})
rows.append({"class": "ALL", "train_inst": split_table.drop(index="images")["train"].sum(),
             "test_inst": split_table.drop(index="images")["test"].sum(),
             "box_P": test_metrics.box.mp, "box_R": test_metrics.box.mr,
             "box_mAP50": test_metrics.box.map50, "box_mAP50-95": test_metrics.box.map,
             "mask_P": test_metrics.seg.mp, "mask_R": test_metrics.seg.mr,
             "mask_mAP50": test_metrics.seg.map50, "mask_mAP50-95": test_metrics.seg.map})
per_class_test = pd.DataFrame(rows).round(3)
per_class_test.to_csv(RUN_DIR / "per_class_test_metrics.csv", index=False)
print("Classes without any test instances are not listed (they cannot be scored).")
per_class_test

**Confusion matrix and precision-recall curve (test set).** In the confusion matrix, the
diagonal holds correct classes. The `background` row/column holds **missed** objects and
**false alarms**. A frequent confusion in this dataset is between a lesion class and the leaf
class of the same crop.

In [ ]:
TEST_DIR = Path(test_metrics.save_dir)
for pattern in ["confusion_matrix_normalized.png", "Mask*PR_curve.png"]:
    for f in sorted(TEST_DIR.glob(pattern))[:1]:
        print(f.name); display(Image(filename=str(f), width=700))

# 10. Predict masks on the test images

The model predicts masks on a few random test images (`predict_one`). Only objects above the confidence threshold
(`CONF`, default 0.25) are kept. A lower threshold finds more lesions (higher recall) but also
gives more false alarms. `retina_masks=True` returns the masks at the full image resolution,
which we need to measure areas in Step 11. The images are
large (about 4000 x 2250 pixels), so only the images shown are predicted here; Step 11 predicts all test
images one at a time.

Below, the **annotation** (left) is compared with the **prediction** (right) for a few random
test images. Look for lesions that were **missed**, **false alarms**, and **outlines** that are too
large or too small.

> **For your report (Results/Discussion):** keep one good example and at least one failure.
> Is the failure a real model error, or is the annotation itself incomplete?

In [ ]:
CONF = 0.25   # confidence threshold for keeping a predicted object
test_imgs = split_imgs["test"]

def predict_one(img_path):
    """Image, full-resolution masks (objects x H x W, bool) and class ids predicted for one image."""
    r = model.predict(img_path, conf=CONF, imgsz=IMGSZ, retina_masks=True, device=DEVICE, verbose=False)[0]
    cls = r.boxes.cls.cpu().numpy().astype(int)
    masks = (np.zeros((0, *r.orig_shape), bool) if r.masks is None
             else r.masks.data.cpu().numpy() > 0.5)
    return r.orig_img, masks, cls

def draw_pred(img_bgr, masks, cls):
    im = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    overlay = im.copy()
    for m, c in zip(masks, cls):
        overlay[m] = PALETTE[c]
        cnts, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(im, cnts, -1, PALETTE[c], max(2, im.shape[1] // 400))
    return cv2.addWeighted(overlay, 0.4, im, 0.6, 0)

# only the images shown here are predicted; Step 11 predicts all test images one at a time
for p in random.sample(test_imgs, min(3, len(test_imgs))):
    img, masks, cls = predict_one(p)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 6))
    a1.imshow(draw_polys(p)); a1.set_title(f"Annotation ({len(read_polys(label_of(p)))} objects)")
    a2.imshow(draw_pred(img, masks, cls)); a2.set_title(f"Prediction ({len(cls)} objects)")
    for a in (a1, a2): a.axis("off")
    plt.suptitle(Path(p).name[:40]); plt.tight_layout(); plt.show()

# 11. Post-analysis: disease severity (% damaged leaf area)

This is where segmentation pays off. Using the masks we compute, for every image:

$$\text{severity (\%)} = \frac{\text{damaged pixels}}{\text{plant pixels}} \times 100$$

- **damaged pixels** = the union of all **disease/pest** masks
- **plant pixels** = the union of all **leaf** masks *and* damage masks. Including the damage
  masks makes the formula correct whether the annotators drew lesions *inside* the leaf outline
  or cut them out of it.

We compute severity twice, from the **human annotations** (the reference) and from the **model
predictions**, and compare the two. This tells us whether the model is good enough to replace a
human rater for severity scoring, which is a different question from mAP.

### 11.1 Which classes are "leaf" and which are "damage"?
A class counts as **leaf** if its name **ends with "leaf"** (`tomato leaf`), and as
**damage** otherwise (`tomato leaf miner`, `potato late blight`). **Check the printed table**; if
a class is assigned wrongly, edit `kind_of`.

In [ ]:
def kind_of(name):
    n = name.lower().strip()
    return "leaf" if (n.endswith("leaf") or "healthy" in n) else "damage"

def crop_of(name):
    return name.replace("_", " ").split()[0].lower()      # "tomato early blight" -> "tomato"

class_kind = {i: kind_of(n) for i, n in enumerate(names)}
pd.DataFrame({"class": names, "crop": [crop_of(n) for n in names],
              "kind": [class_kind[i] for i in range(len(names))]})

### 11.2 Severity per test image: annotation vs prediction
For each test image, one at a time, the model predicts the masks (`predict_one`), and the annotated
polygons are filled in to get the reference masks. From these we compute the two severity values;
only these numbers are kept, so memory use stays low. The table also records the crop (from the annotation) and the absolute
error.

In [ ]:
def severity(leaf, damage):
    plant = leaf | damage
    return 100 * damage.sum() / plant.sum() if plant.any() else np.nan

def gt_severity(img_path, shape):
    H, W = shape
    leaf, damage = np.zeros((H, W), np.uint8), np.zeros((H, W), np.uint8)
    polys = read_polys(label_of(img_path))
    for cls, pts in polys:
        cv2.fillPoly(leaf if class_kind[cls] == "leaf" else damage,
                     [(pts * [W, H]).round().astype(np.int32)], 1)
    crops = Counter(crop_of(names[c]) for c, _ in polys)
    return severity(leaf.astype(bool), damage.astype(bool)), (crops.most_common(1)[0][0] if crops else "none")

def pred_severity(masks, cls):
    H, W = masks.shape[1:]
    leaf, damage = np.zeros((H, W), bool), np.zeros((H, W), bool)
    for m, c in zip(masks, cls):
        if class_kind[c] == "leaf": leaf |= m
        else: damage |= m
    return severity(leaf, damage)

# predict the test images one at a time and keep only the two severity values,
# so the full-resolution images and masks never pile up in memory
rows = []
for p in test_imgs:
    img, masks, cls = predict_one(p)
    gt, crop = gt_severity(p, img.shape[:2])
    rows.append({"image": Path(p).name, "crop": crop,
                 "gt_severity": gt, "pred_severity": pred_severity(masks, cls)})
sev = pd.DataFrame(rows)
sev["abs_error"] = (sev["pred_severity"] - sev["gt_severity"]).abs()
sev.to_csv(RUN_DIR / "severity_test.csv", index=False)
print(f"{len(sev)} test images | no annotated plant: {sev['gt_severity'].isna().sum()} | "
      f"no predicted plant: {sev['pred_severity'].isna().sum()}")
sev.round(2).sort_values("gt_severity", ascending=False).head(10)

### 11.3 How well does the model estimate severity?
Summary per crop and overall (images where both values exist):
- `gt_mean` / `pred_mean`: average severity from the annotation vs the model.
- `MAE`: mean absolute error in **percentage points** (e.g. MAE = 3 means the model is off by
  3 % leaf area on average).
- `bias`: mean of (predicted − annotated). Negative = the model **under-estimates** severity
  (misses lesions), positive = it **over-estimates**.
- `r`: Pearson correlation. Close to 1 means the model ranks images by severity the same way the
  annotation does, which is what matters for, e.g., comparing varieties.

The scatter plot shows each test image: points on the dashed line are perfect estimates.

> **For your report (Results/Discussion):** include the summary table and the scatter plot. Is
> the error small compared with the typical severity values? Does the model systematically under-
> or over-estimate? Is it equally good for all crops?

In [ ]:
ok = sev.dropna(subset=["gt_severity", "pred_severity"])

def summarise(d):
    return pd.Series({"images": len(d), "gt_mean": d["gt_severity"].mean(),
                      "pred_mean": d["pred_severity"].mean(), "MAE": d["abs_error"].mean(),
                      "bias": (d["pred_severity"] - d["gt_severity"]).mean(),
                      "r": d["gt_severity"].corr(d["pred_severity"]) if len(d) > 2 else np.nan})

sev_summary = pd.concat([ok.groupby("crop").apply(summarise),
                         summarise(ok).to_frame("ALL").T]).round(2)
sev_summary.to_csv(RUN_DIR / "severity_summary.csv")

fig, ax = plt.subplots(figsize=(6.5, 6))
for crop, d in ok.groupby("crop"):
    ax.scatter(d["gt_severity"], d["pred_severity"], alpha=0.7, label=crop)
mx = max(ok["gt_severity"].max(), ok["pred_severity"].max(), 1) * 1.05 if len(ok) else 100
ax.plot([0, mx], [0, mx], "k--", lw=1, label="perfect estimate")
ax.set_xlabel("severity from annotation (%)"); ax.set_ylabel("severity from model (%)")
ax.set_title("Disease severity per test image"); ax.legend()
plt.tight_layout(); plt.savefig(RUN_DIR / "severity_scatter.png", dpi=150); plt.show()
sev_summary

# 12. Log this experiment

Every time you run the notebook with different settings, the cell below adds **one row** to
`/content/runs/plant_seg/experiment_log.csv`. After a few runs, this table is the basis for your
*experiment* table in the report.

> **Note:** Colab deletes `/content` when the session ends. Download the log, the figures
> (`results.png`, confusion matrix, `severity_scatter.png`), the CSV tables and `best.pt` (Files
> panel, right-click, *Download*), or copy them to Google Drive.

In [ ]:
log_path = Path(PROJECT) / "experiment_log.csv"
entry = pd.DataFrame([{
    "run": RUN_DIR.name, "crops": crops_tag, "model": MODEL_NAME, "epochs_max": EPOCHS,
    "epochs_run": len(hist), "best_epoch": int(best_row["epoch"]), "imgsz": IMGSZ, "batch": BATCH,
    "val_mask_mAP50": round(val_metrics.seg.map50, 3),
    "test_box_mAP50": round(test_metrics.box.map50, 3),
    "test_mask_mAP50": round(test_metrics.seg.map50, 3),
    "test_mask_mAP50-95": round(test_metrics.seg.map, 3),
    "severity_MAE": sev_summary.loc["ALL", "MAE"], "severity_r": sev_summary.loc["ALL", "r"],
}])
log = pd.concat([pd.read_csv(log_path), entry]) if log_path.exists() else entry
log = log.drop_duplicates(subset="run", keep="last")
log.to_csv(log_path, index=False)
print("Saved", log_path)
log

# 13. Summary

In this notebook you:
- inspected a polygon-annotated plant-disease dataset and built a leakage-free, reproducible
  train/validation/test split;
- fine-tuned a pretrained YOLO **instance segmentation** model on leaves and disease/pest damage;
- evaluated it with **box and mask** precision, recall and mAP on the validation and **test**
  sets, overall and per class;
- used the predicted masks to estimate **disease severity** (% damaged leaf area) and compared it
  with the severity from the human annotations;
- logged the experiment.

**Ideas for improving results:** train longer, use a larger model (`yolo26s-seg.pt`), or use a
larger `IMGSZ` so small lesions keep enough pixels (lower `BATCH` to fit memory). You can also
train one model per crop (Step 4) and compare with a single model for all crops.

# Assignment: write your report

You now have everything you need. Your task is to **get the best results you can** by making
small changes to the settings in **Step 7.1** (e.g. `EPOCHS`, `MODEL_NAME`, `IMGSZ`, and
optionally the crop selection in Step 4), and then to describe the study in a **written report
of about 2 pages**. You will also present your findings to the class in a short **PowerPoint
presentation** built from the same material.

### Before you write
1. Run the notebook once with the default settings. This is your **baseline**.
2. Run **at least 2 more experiments**, changing **one setting at a time** so you can tell what
   caused any difference. Each run is added to `experiment_log.csv` (Step 12).
3. **Download** your log, tables, figures and best weights before the Colab session ends.

### Report structure

**1. Introduction (~¼ page)**
- The plant-pathology problem: the crops, the diseases/pests in the dataset and why they matter.
- Why **disease severity** is measured, and the limitations of visual estimation by human raters.
- What instance segmentation is and why it is needed to measure area (vs classification and
  detection).
- **The objective of your study**, stated clearly in the last sentence of the introduction.

**2. Methodology (~½ page)**: give enough detail that someone else could repeat your work.
- **Dataset:** source, number of images, crops, classes, instances per class, crop selection
  (if any).
- **Data preparation:** polygon labels, the train/validation/test split (how it was made,
  sizes, why augmented copies were grouped).
- **Model:** YOLO segmentation architecture and size, pretrained weights and **transfer learning**.
- **Training settings:** epochs, early-stopping patience, image size, batch size, hardware (GPU).
- **Evaluation:** precision, recall, IoU, **box vs mask mAP50 / mAP50-95** (define each in one
  sentence), and how severity was calculated (formula, leaf vs damage classes, MAE and
  correlation against the annotations).

**3. Results (~½-¾ page)**: report *what you observed*; leave the interpretation for the
discussion. You can use tables and figures, or describe the results in short sentences. Number every
table and figure, give each a caption, and refer to it in the text.

**4. Discussion and conclusion (~½ page)**: interpret your results. You do not have to answer all of these questions; use them as a guide:
- Which classes are segmented well and which poorly? Relate this to **object size** (leaves vs
  small lesions or mines), the **number of training instances**, and how **visually distinct**
  the symptoms are.
- Why is **mask** mAP lower than **box** mAP, and mAP50-95 lower than mAP50?
- How close are the model's **severity** estimates to the annotation-based values? Does it under-
  or over-estimate, and why (missed small lesions, leaf outlines too small, ...)? Would the
  accuracy be good enough for a breeding trial or a spray decision?
- What did changing the epochs / model size / image size / crop selection do? Did training show
  **over-fitting**? Did early stopping trigger?
- **Limitations:** the annotations are the "truth", but annotators can miss lesions or draw rough
  outlines. Severity here is *per photo*, not per leaf. There was a single split, the image
  conditions may be limited, and some classes have few instances.
- What would you do next (more images of the rare classes, larger image size, per-crop models,
  field validation against an expert rater)?

End with a short **conclusion** (2-3 sentences) that directly answers the objective stated in
your introduction, and suggest one or two concrete next steps.

**5. References**: cite the dataset (Kaggle link), Ultralytics YOLO
(https://docs.ultralytics.com) and any plant-pathology sources used in the introduction.

If anything in the notebook does not run, contact the instructors. We are happy to help.